# VHR Imagery Foundation: Working with Maxar Open Data
This notebook covers the foundations for working with Very-High-Resolution (VHR) optical satellite imagery from the Maxar Open Data Program. You'll learn to:
1. Set up the environment with geospatial Python libraries.
2. Access and explore the Maxar Open Data catalog.
3. Load and display VHR imagery from cloud-optimized GeoTIFFs.
4. Use window-reading techniques for large imagery files.
5. Characterize clouds, smoke, and cloud-like surfaces in VHR scenes.
6. Assemble a working set of scenes with higher cloud cover.

**Deliverable:** A notebook that loads, displays, and tiles VHR imagery, plus a written characterization of what clouds, smoke, and cloud-like surfaces look like in these scenes.

## 1. Environment Setup

In [ ]:
# Install required packages (run this first time).
!pip install rasterio leafmap pystac-client matplotlib numpy scipy
import libraries
import rasterio
from rasterio.windows import Window
from rasterio.plot import show
import matplotlib.pyplot as plt
import numpy as np
import pystac_client
from pystac_client import Client
import leafmap
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')
print("Libraries imported successfully!")

## 2. Accessing the Maxar Open Data Catalog
The Maxar Open Data Program provides free access to high-resolution satellite imagery for significant events (natural disasters, humanitarian crises, etc.). We'll use the STAC (SpatioTemporal Asset Catalog) API to explore available imagery.

In [ ]:
# Connect to the Maxar Open Data STAC catalog
catalog_url = "https://maxar-opendata.s3.amazonaws.com/events/catalog.json"
catalog = Client.open(catalog_url)
print(f"Connected to catalog: {catalog.title}")
print(f"Catalog description: {catalog.description}")

In [ ]:
# Explore available collections (events)
collections = list(catalog.get_collections())
print(f"Number of available collections: {len(collections)}")
print("\nFirst 10 collections:")
for i, collection in enumerate(collections[:10]):
    print(f"{i+1}. {collection.id}: {collection.title if collection.title else 'No title'}")

In [ ]:
# Let's explore a specific collection (e.g., a hurricane event)
# You can choose different events from the list above
selected_collection_id = collections[0].id  # Change this to explore different events
print(f"Exploring collection: {selected_collection_id}")
collection = catalog.get_collection(selected_collection_id)
print(f"Collection extent: {collection.extent.spatial.bboxes}")
print(f"Collection temporal extent: {collection.extent.temporal}")

In [ ]:
# Get items (individual image scenes) from the collection
items = list(collection.get_items())
print(f"Number of items in this collection: {len(items)}")
# Display information about the first few items
print("\nFirst 5 items:")
for i, item in enumerate(items[:5]):
    print(f"{i+1}. {item.id}")
    print(f"   Geometry: {item.geometry['type']}")
    print(f"   Datetime: {item.datetime}")
    print(f"   Properties: {item.properties.get('eo:cloud_cover', 'N/A')}% cloud cover")
    print()

## 3. Loading and Displaying VHR Imagery
VHR imagery files are often very large (gigabytes). We'll use window-reading techniques to load specific regions of interest rather than entire files.

In [ ]:
# Select an item to work with
selected_item = items[0]  # You can change this index to explore different scenes
print(f"Selected item: {selected_item.id}")
# Get the asset (image file) - typically the 'visual' asset for RGB imagery
assets = selected_item.assets
print("Available assets:")
for asset_key, asset in assets.items():
    print(f"  {asset_key}: {asset.title if asset.title else asset_key}")

In [ ]:
# Get the visual (RGB) asset URL
visual_asset = assets.get('visual')
if visual_asset is None:
# Fallback to other common asset names
    for asset_key in ['rgb', 'image', 'cog']:
        if asset_key in assets:
            visual_asset = assets[asset_key]
            break
            if visual_asset:
                image_url = visual_asset.href
                print(f"Image URL: {image_url}")
                print(f"Image type: {visual_asset.type}")
                print(f"Image roles: {visual_asset.roles}")
            else:
                print("No visual asset found. Available assets:")
                for key in assets.keys():
                    print(f"  {key}")


In [ ]:
# Open the image using rasterio (cloud-optimized GeoTIFF)
with rasterio.open(image_url) as src:
    print(f"Image dimensions: {src.width} x {src.height}")
    print(f"Number of bands: {src.count}")
    print(f"Coordinate reference system: {src.crs}")
    print(f"Transform: {src.transform}")
    print(f"Bounds: {src.bounds}")
    # Check if it's cloud-optimized
    print(f"Is cloud-optimized: {src.is_cloud_optimized}")